# 🛰️ Copernicus Sentinel-2 Data Acquisition + Roof Inference Pipeline

**Author:** Khiw Nitithadachot · **Stack:** sentinelhub-py + rasterio + shapely + ONNX Runtime
**Source:** Copernicus Data Space Ecosystem (CDSE) Sentinel Hub APIs
**Output:** Per-polygon Sentinel-2 L2A tile + ML inference (material seg + corrosion mask)

This notebook is the production data plane for the FloodSight/CarbonScope/roof-corrosion stack:

```
   user polygon (GeoJSON)
        │
        ▼
   CDSE OAuth2 (Keycloak)
        │
        ▼
   Sentinel Hub Catalog API     ←  find best cloud-free L2A scene
        │
        ▼
   Sentinel Hub Process API     ←  reproject, clip, cloud-mask, return tile
        │
        ▼
   roof_tile.tif (384x384, 13 bands)
        │
        ▼
   ONNX Runtime  + clay_*.onnx ←  the model from the training notebook
        │
        ▼
   {material_mask, corrosion_mask, severity, area_m²}
```

---

## ⚠️ Before running: rotate your CDSE credentials

If you ever pasted credentials into chat, source code, a screenshot, or a public notebook, **revoke them now**:

1. https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings
2. Delete the exposed client.
3. Create a new client with OAuth2 + the SH service.
4. Save the new `client_id` and `client_secret` to **Kaggle Secrets** (`Add-ons → Secrets`):
   - `CDSE_CLIENT_ID`
   - `CDSE_CLIENT_SECRET`
5. Never commit them to git, never paste them anywhere.

This notebook reads ONLY from Kaggle Secrets / environment variables. There is no fallback to inline credentials.

## ⚠️ Sentinel-2 reality check (one more time)

Sentinel-2 is **10 m native resolution**. A 100 m² roof = 1 pixel. This pipeline is the **Tier 0 free preliminary** layer of the architecture: useful for *building presence + coarse material*, **not** for billable corrosion area. For binding quotes, escalate to Pléiades archive (€3.80/km²) or drone capture as documented in the architecture deep-dive.

The pipeline ships imagery with **all 13 bands** so the model can use spectral indices (Iron Oxide Ratio, BCCSR) — these compensate partially for the resolution gap on metal roofs ≥200 m².

## 1. Environment setup

In [29]:
%%capture
!pip install -q -U \
    "sentinelhub==3.10.4" \
    "oauthlib==3.2.2" \
    "requests-oauthlib==2.0.0" \
    "rasterio==1.4.2" \
    "geopandas==1.0.1" \
    "shapely==2.0.6" \
    "pyproj==3.7.0" \
    "fiona==1.10.1" \
    "folium==0.18.0" \
    "tqdm==4.67.1" \
    "onnxruntime-gpu==1.20.1" \
    "opencv-python-headless==4.10.0.84"


In [30]:
!pip install sentinelhub

In [31]:
import os, sys, json, math, time, gc, warnings
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Load credentials from .env.local if running locally
load_dotenv(Path.cwd().parent / ".env.local")
warnings.filterwarnings("ignore")

import numpy as np
import rasterio
from rasterio.transform import from_bounds
from shapely.geometry import shape, mapping, box, Polygon, MultiPolygon
from shapely.ops import transform as shp_transform
import pyproj
from pyproj import Transformer

# --- Credentials from Kaggle Secrets (NEVER inline) -------------------------
def get_cdse_credentials():
    cid = secret = None
    try:
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        try:
            cid = sec.get_secret("CDSE_CLIENT_ID")
            secret = sec.get_secret("CDSE_CLIENT_SECRET")
        except Exception:
            pass
    except Exception:
        pass
    cid = cid or os.environ.get("CDSE_CLIENT_ID")
    secret = secret or os.environ.get("CDSE_CLIENT_SECRET")
    if not (cid and secret):
        raise RuntimeError(
            "CDSE credentials not found.\n"
            "Add to Kaggle Secrets: CDSE_CLIENT_ID, CDSE_CLIENT_SECRET\n"
            "Or export as env vars before launching the notebook."
        )
    # Quick sanity check WITHOUT echoing the secret
    print(f"✓ CDSE creds loaded (client_id starts with '{cid[:8]}…', secret length={len(secret)})")
    return cid, secret

CDSE_ID, CDSE_SECRET = get_cdse_credentials()


✓ CDSE creds loaded (client_id starts with 'sh-5dce7…', secret length=32)


## 2. CDSE Sentinel Hub configuration

CDSE Sentinel Hub uses a separate set of endpoints from the legacy commercial Sentinel Hub. We point `sentinelhub-py`'s `SHConfig` at CDSE explicitly.

In [32]:
from sentinelhub import SHConfig, DataCollection, BBox, CRS, MimeType
from sentinelhub import SentinelHubRequest, SentinelHubCatalog, MosaickingOrder
from sentinelhub.api.process import SentinelHubRequest

# CDSE Sentinel Hub endpoints (NOT the commercial sh.com URLs)
CDSE_AUTH_URL    = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
CDSE_BASE_URL    = "https://sh.dataspace.copernicus.eu"
CDSE_CATALOG_URL = "https://sh.dataspace.copernicus.eu/api/v1/catalog"

config = SHConfig()
config.sh_client_id     = CDSE_ID
config.sh_client_secret = CDSE_SECRET
config.sh_token_url     = CDSE_AUTH_URL
config.sh_base_url      = CDSE_BASE_URL
# don't save() — keeps secrets out of ~/.config/sentinelhub/

print("Endpoint:", config.sh_base_url)
print("Token URL:", config.sh_token_url)


Endpoint: https://sh.dataspace.copernicus.eu
Token URL: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token


In [33]:
# --- Auth smoke test --------------------------------------------------------
import requests
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

def get_cdse_token(client_id: str, client_secret: str) -> dict:
    """Standalone OAuth2 client_credentials flow against CDSE Keycloak.
    Returns the token dict; access_token is valid ~10 min."""
    client = BackendApplicationClient(client_id=client_id)
    oauth = OAuth2Session(client=client)
    token = oauth.fetch_token(
        token_url=CDSE_AUTH_URL,
        client_secret=client_secret,
        include_client_id=True,
    )
    return token

try:
    tok = get_cdse_token(CDSE_ID, CDSE_SECRET)
    print(f"✓ Token acquired. expires_in={tok['expires_in']}s, type={tok['token_type']}")
    # Reveal nothing about the secret in logs
    print(f"  access_token: {tok['access_token'][:12]}…{tok['access_token'][-6:]}")
except Exception as e:
    print(f"✗ Auth failed: {type(e).__name__}: {e}")
    print("  Check: (a) credentials valid, (b) account has SH service activated at")
    print("         https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings")
    raise


✓ Token acquired. expires_in=1800s, type=Bearer
  access_token: eyJhbGciOiJS…ZPaMyw


## 3. Define an AOI polygon

Three input modes:

1. **GeoJSON file** at `/kaggle/input/<dataset>/aoi.geojson` (single Feature or FeatureCollection)
2. **Inline GeoJSON** (paste from geojson.io)
3. **Bbox helper** for quick tests

This notebook follows the customer-quotation user journey: customer submits a polygon → we fetch satellite data for that polygon. For batch training data, swap a multi-polygon FeatureCollection for whole districts.

In [34]:
# --- Example AOI: a small block in Bangkok with metal-sheet roof factories ---
# Replace with your own GeoJSON. Bangkok industrial north (Lat Krabang) coords.
EXAMPLE_AOI = {
    "type": "Feature",
    "properties": {"name": "Bangkok-LatKrabang-test"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [100.7748, 13.7220],
            [100.7848, 13.7220],
            [100.7848, 13.7320],
            [100.7748, 13.7320],
            [100.7748, 13.7220],
        ]],
    },
}

# Load: prefer file, fall back to example
AOI_FILE = Path("/kaggle/input/aoi/aoi.geojson")
if AOI_FILE.exists():
    with open(AOI_FILE) as f:
        gj = json.load(f)
    if gj["type"] == "FeatureCollection":
        feature = gj["features"][0]
    else:
        feature = gj
    print(f"✓ Loaded AOI from {AOI_FILE}")
else:
    feature = EXAMPLE_AOI
    print("ℹ Using built-in Bangkok-LatKrabang example AOI")

aoi_geom = shape(feature["geometry"])
aoi_name = feature.get("properties", {}).get("name", "aoi")
print(f"  name: {aoi_name}")
print(f"  area: {aoi_geom.area * 111**2 * 1e6:.0f} m² (≈ {aoi_geom.area * 111**2:.4f} km²)")
print(f"  bbox: {aoi_geom.bounds}")


ℹ Using built-in Bangkok-LatKrabang example AOI
  name: Bangkok-LatKrabang-test
  area: 1232100 m² (≈ 1.2321 km²)
  bbox: (100.7748, 13.722, 100.7848, 13.732)


In [35]:
# --- Visualise the AOI on a folium map (tiles from CARTO, no API key) -------
import folium
m = folium.Map(location=[(aoi_geom.bounds[1]+aoi_geom.bounds[3])/2,
                          (aoi_geom.bounds[0]+aoi_geom.bounds[2])/2], zoom_start=16,
               tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
               attr="© CARTO © OpenStreetMap contributors")
folium.GeoJson(feature, name=aoi_name).add_to(m)
folium.LatLngPopup().add_to(m)
m


## 4. Catalog search — find the best recent cloud-free L2A scene

Two-step strategy:
1. **Catalog API** — list all S2 L2A scenes intersecting the AOI in the last 90 days.
2. **Cloud cover filter** — pick the scene with `eo:cloud_cover < 20%` AND most recent acquisition.

If no scene clears 20% cloud (common in Thailand monsoon May–Oct), fall back to:
- 60-day window
- Best-Available-Pixel composite (covered in Section 6)
- S1 SAR gap-fill (a separate notebook)

In [36]:
from sentinelhub import SentinelHubCatalog
from datetime import datetime, timedelta, timezone

catalog = SentinelHubCatalog(config=config)

def search_s2_scenes(aoi_geom, days_back: int = 90, max_cloud: float = 20.0):
    """Returns list of catalog entries sorted by date desc, filtered by cloud cover."""
    end   = datetime.now(timezone.utc)
    start = end - timedelta(days=days_back)
    bbox  = BBox(bbox=aoi_geom.bounds, crs=CRS.WGS84)

    iter_results = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=bbox,
        time=(start, end),
        filter=f"eo:cloud_cover<{max_cloud}",
        fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover",
                            "properties.s2:mgrs_tile", "properties.platform"]},
    )
    results = list(iter_results)
    return sorted(results, key=lambda x: x["properties"]["datetime"], reverse=True)

print(f"Searching CDSE Catalog for S2 L2A scenes intersecting AOI…")
scenes = search_s2_scenes(aoi_geom, days_back=90, max_cloud=20)
print(f"Found {len(scenes)} scenes with <20% cloud in last 90 days\n")
for s in scenes[:8]:
    p = s["properties"]
    print(f"  {p['datetime'][:10]} | tile {p.get('s2:mgrs_tile', '?')} | "
          f"cloud {p['eo:cloud_cover']:5.1f}% | {p.get('platform', 's2')} | id={s['id']}")

if not scenes:
    print("⚠ No scenes — widen window or raise cloud threshold")
    scenes = search_s2_scenes(aoi_geom, days_back=180, max_cloud=60)
    print(f"  Fallback search returned {len(scenes)} scenes (≤60% cloud, 180d)")

assert scenes, "No S2 scenes available — check AOI is on land and CDSE service is up"
best_scene = scenes[0]
print(f"\n→ Selected: {best_scene['properties']['datetime'][:10]} "
      f"({best_scene['properties']['eo:cloud_cover']:.1f}% cloud)")


Searching CDSE Catalog for S2 L2A scenes intersecting AOI…


ValueError: Data collection definition is already taken by DataCollection.s2l2a_cdse. Two different DataCollection enums cannot have the same definition.

## 5. Process API — fetch a multispectral GeoTIFF

We pull all the bands the model needs:
- **B02, B03, B04** (10 m, RGB)
- **B05, B06, B07** (20 m, red edge — for material discrimination)
- **B08** (10 m, NIR)
- **B8A** (20 m, narrow NIR)
- **B11, B12** (20 m, SWIR — iron oxide / BCCSR / clay indices)
- **SCL** (20 m, Scene Classification — Sen2Cor cloud/shadow mask)

Sen2Cor L2A is server-side, so we get **bottom-of-atmosphere reflectance** values directly — critical for spectral index calculations. The evalscript downsamples 20m bands to 10m so all bands stack cleanly.

### Output sizing for the model

Our trained model expects **384×384 input**. We size the SH request to produce exactly that:
- AOI bbox in WGS84 → reproject to UTM zone (Bangkok = EPSG:32647)
- Compute true ground span in metres
- Request `width=384, height=384` → SH resamples to fit

In [ ]:
# --- Pick the right UTM zone for the AOI ------------------------------------
def utm_epsg_for_geom(geom):
    """Return EPSG code for the appropriate UTM zone (north hemisphere, Thailand)."""
    cx, cy = geom.centroid.x, geom.centroid.y
    zone = int((cx + 180) / 6) + 1
    return 32600 + zone if cy >= 0 else 32700 + zone

epsg_utm = utm_epsg_for_geom(aoi_geom)
print(f"AOI centroid: ({aoi_geom.centroid.x:.4f}, {aoi_geom.centroid.y:.4f})")
print(f"UTM zone EPSG: {epsg_utm}")

# Reproject AOI to UTM to get accurate metres
to_utm   = Transformer.from_crs(4326, epsg_utm, always_xy=True).transform
to_wgs   = Transformer.from_crs(epsg_utm, 4326, always_xy=True).transform
aoi_utm  = shp_transform(to_utm, aoi_geom)
minx, miny, maxx, maxy = aoi_utm.bounds
ground_w_m = maxx - minx
ground_h_m = maxy - miny
print(f"Ground extent: {ground_w_m:.0f} × {ground_h_m:.0f} m")
print(f"S2 native:     {ground_w_m/10:.0f} × {ground_h_m/10:.0f} pixels @ 10m")


AOI centroid: (100.7798, 13.7270)
UTM zone EPSG: 32647
Ground extent: 1090 × 1114 m
S2 native:     109 × 111 pixels @ 10m


In [ ]:
# --- Evalscript: emit 10 reflectance bands + SCL as multi-band TIFF ---------------------
EVALSCRIPT_S2_FULL = """
//VERSION=3
function setup() {
  return {
    input: [
      {
        bands: ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"],
        units: "REFLECTANCE"
      },
      {
        bands: ["SCL"],
        units: "DN"
      }
    ],
    output: { bands: 11, sampleType: "FLOAT32" },
    mosaicking: "ORBIT"
  };
}

function evaluatePixel(sample) {
  return [sample.B02, sample.B03, sample.B04, sample.B05, sample.B06, sample.B07, sample.B08, sample.B8A, sample.B11, sample.B12, sample.SCL];
}
"""

# Output size: model expects 384×384
OUT_W, OUT_H = 384, 384


In [ ]:
print("Process URL:", config.sh_base_url)
print("Token URL:  ", config.sh_token_url)

In [ ]:
# --- Fixed Sentinel‑Hub Process API request ---------------------------------
from sentinelhub import SentinelHubRequest, DataCollection, MimeType, MosaickingOrder

request = SentinelHubRequest(
    evalscript=EVALSCRIPT_S2_FULL,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,   # ← corrected
            time_interval=time_window,
            mosaicking_order=MosaickingOrder.LEAST_CC,
            maxcc=0.6,
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=bbox_utm,
    size=(OUT_W, OUT_H),
    config=config,
)

print(f"Requesting Sentinel‑2 L2A tile…")
print(f"  bbox UTM: {bbox_utm}")
print(f"  size: {OUT_W}×{OUT_H}")
print(f"  date: {acquisition_date}")

t0 = time.time()
data = request.get_data()[0]   # numpy array (H, W, 11)
print(f"✓ Received in {time.time()-t0:.1f}s")
print(f"  shape: {data.shape} dtype: {data.dtype}")
print(f"  reflectance range B04: {data[...,2].min():.3f} – {data[...,2].max():.3f}")


NameError: name 'EVALSCRIPT_S2_FULL' is not defined

In [ ]:
# --- Save as proper georeferenced GeoTIFF for downstream tools --------------
OUT_DIR = Path("/kaggle/working/cdse_tiles")
OUT_DIR.mkdir(parents=True, exist_ok=True)
tile_path = OUT_DIR / f"{aoi_name}_{acquisition_date}_s2l2a.tif"

# Bands order matches the evalscript output
band_names = ["B02", "B03", "B04", "B05", "B06", "B07",
              "B08", "B8A", "B11", "B12", "SCL"]

transform = from_bounds(minx, miny, maxx, maxy, OUT_W, OUT_H)
with rasterio.open(
    tile_path, "w",
    driver="COG",
    width=OUT_W, height=OUT_H, count=len(band_names),
    dtype="float32", crs=f"EPSG:{epsg_utm}",
    transform=transform,
    compress="zstd", blocksize=256, overviews="auto",
) as dst:
    for i, name in enumerate(band_names):
        dst.write(data[..., i].astype("float32"), i + 1)
        dst.set_band_description(i + 1, name)
    dst.update_tags(
        acquisition_date=acquisition_date,
        cloud_cover=str(best_scene["properties"]["eo:cloud_cover"]),
        scene_id=best_scene["id"],
        aoi_name=aoi_name,
    )

print(f"✓ Saved {tile_path} ({tile_path.stat().st_size/1e6:.1f} MB)")


## 6. Quality control + spectral indices

### SCL classification (Sen2Cor)

| Code | Class | Action |
|---|---|---|
| 0 | NO_DATA | reject |
| 1 | SATURATED_OR_DEFECTIVE | reject |
| 2 | CAST_SHADOWS | mask, downweight |
| 3 | CLOUD_SHADOWS | reject |
| 4 | VEGETATION | keep |
| 5 | NOT_VEGETATED (built-up, soil) | **keep — primary roof signal** |
| 6 | WATER | keep |
| 7 | UNCLASSIFIED | keep with warning |
| 8 | CLOUD_MEDIUM_PROBABILITY | reject |
| 9 | CLOUD_HIGH_PROBABILITY | reject |
| 10 | THIN_CIRRUS | mask, downweight |
| 11 | SNOW_OR_ICE | not relevant in Thailand |

**Don't trust SCL alone on bright zinc roofs in Thailand — it over-calls them as cloud.** We use the 2-of-3 rule: SCL says cloud AND OmniCloudMask says cloud AND cloud probability > 60%. For now, simple SCL filter; OmniCloudMask integration is a 30-line add-on.

In [ ]:
# --- SCL → valid pixel mask -------------------------------------------------
SCL_VALID = {4, 5, 6, 7}    # vegetation, built-up, water, unclassified
SCL_REJECT = {0, 1, 3, 8, 9, 10, 11}    # no-data, defective, shadows, clouds

scl = data[..., 10].astype(np.uint8)
valid_mask = np.isin(scl, list(SCL_VALID))
reject_mask = np.isin(scl, list(SCL_REJECT))
n_total = scl.size
n_valid = valid_mask.sum()
print(f"SCL summary:")
print(f"  valid pixels:  {n_valid:>7}/{n_total} ({100*n_valid/n_total:.1f}%)")
print(f"  rejected:      {reject_mask.sum():>7}/{n_total} ({100*reject_mask.sum()/n_total:.1f}%)")
for code in sorted(np.unique(scl)):
    n = (scl == code).sum()
    if n == 0: continue
    label = ["NO_DATA","SAT/DEF","CAST_SHADOW","CLOUD_SHADOW","VEG","NOT_VEG",
             "WATER","UNCLASS","CLOUD_MED","CLOUD_HIGH","CIRRUS","SNOW"][code]
    print(f"    {code:2d} {label:14}: {n:>7} px ({100*n/n_total:5.1f}%)")

if n_valid / n_total < 0.5:
    print("\n⚠ More than half the tile is rejected. Consider: BAP composite or escalate to VHR.")


In [ ]:
# --- Spectral indices for material classification ---------------------------
# Reflectance-aware (data is already in 0-1 reflectance from the L2A request)
B = {n: data[..., i].astype(np.float32) for i, n in enumerate(band_names)}

def safe_div(a, b, eps=1e-6): return a / (b + eps)

indices = {
    # General
    "ndvi":       safe_div(B["B08"] - B["B04"], B["B08"] + B["B04"]),
    "ndwi":       safe_div(B["B03"] - B["B08"], B["B03"] + B["B08"]),
    "ndbi":       safe_div(B["B11"] - B["B08"], B["B11"] + B["B08"]),
    # Iron / rust signatures
    "iron_oxide_ratio": safe_div(B["B04"], B["B02"]),     # Liu 2020 — strongest rust proxy
    "ferrous_iron_idx": safe_div(B["B11"], B["B8A"]),
    "iron_mixture":     safe_div(B["B06"] + B["B07"], B["B8A"]),
    # Roof material
    "bccsr":      safe_div(B["B02"] - B["B11"], B["B02"] + B["B11"]),  # Hou 2023 painted/coated steel
    "ndmci_port": safe_div(B["B04"] - B["B11"], B["B04"] + B["B11"]),
    "clay_minerals": safe_div(B["B11"], B["B12"]),
}

print(f"Computed {len(indices)} spectral indices:")
for k, v in indices.items():
    valid_v = v[valid_mask]
    if valid_v.size > 0:
        print(f"  {k:18}: mean={valid_v.mean():+.3f}  std={valid_v.std():.3f}  "
              f"p5/p95={np.percentile(valid_v, 5):+.3f}/{np.percentile(valid_v, 95):+.3f}")


In [ ]:
# --- Persist indices alongside the raw bands as additional channels ---------
indices_out = OUT_DIR / f"{aoi_name}_{acquisition_date}_indices.tif"
idx_arr = np.stack(list(indices.values()), axis=0).astype("float32")    # (n_idx, H, W)
with rasterio.open(
    indices_out, "w",
    driver="COG",
    width=OUT_W, height=OUT_H, count=len(indices),
    dtype="float32", crs=f"EPSG:{epsg_utm}",
    transform=transform,
    compress="zstd", blocksize=256,
) as dst:
    for i, (name, arr) in enumerate(indices.items()):
        dst.write(arr, i + 1)
        dst.set_band_description(i + 1, name)
print(f"✓ Indices saved → {indices_out} ({indices_out.stat().st_size/1e6:.1f} MB)")


## 7. Visualise the tile + indices

In [ ]:
import matplotlib.pyplot as plt

# True colour (B04, B03, B02) with linear stretch
def stretch(rgb, p_low=2, p_high=98):
    out = np.zeros_like(rgb, dtype=np.float32)
    for c in range(3):
        ch = rgb[..., c]
        lo, hi = np.percentile(ch, [p_low, p_high])
        out[..., c] = np.clip((ch - lo) / (hi - lo + 1e-9), 0, 1)
    return out

rgb = np.stack([B["B04"], B["B03"], B["B02"]], axis=-1)
rgb_disp = stretch(rgb)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes[0, 0].imshow(rgb_disp); axes[0, 0].set_title(f"True colour\n{acquisition_date}")
axes[0, 1].imshow(scl, cmap="tab20", vmin=0, vmax=11); axes[0, 1].set_title("SCL")
axes[0, 2].imshow(valid_mask, cmap="RdYlGn"); axes[0, 2].set_title(f"Valid mask ({100*n_valid/n_total:.0f}%)")
axes[0, 3].imshow(indices["ndvi"], cmap="RdYlGn", vmin=-0.3, vmax=0.8); axes[0, 3].set_title("NDVI")

axes[1, 0].imshow(indices["ndbi"], cmap="RdBu_r", vmin=-0.5, vmax=0.5); axes[1, 0].set_title("NDBI (built-up)")
axes[1, 1].imshow(indices["iron_oxide_ratio"], cmap="hot", vmin=0.5, vmax=2.0); axes[1, 1].set_title("Iron Oxide Ratio")
axes[1, 2].imshow(indices["bccsr"], cmap="coolwarm", vmin=-0.4, vmax=0.4); axes[1, 2].set_title("BCCSR (painted steel)")
axes[1, 3].imshow(indices["ferrous_iron_idx"], cmap="hot", vmin=0.5, vmax=2.5); axes[1, 3].set_title("Ferrous Iron Idx")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()


## 8. Run the trained model on this tile

Loads the ONNX export from the training notebook and produces:
- **material_mask** (5-class: corrugated/painted/standing-seam/metal-tile/other)
- **corrosion_mask** (binary)
- **severity_class** (none/light/moderate/severe)

### Loading the ONNX model

Three options, in order of preference:

1. **Kaggle Dataset attachment** — upload `clay_*.onnx` from the training notebook as a Kaggle Dataset, attach here. Path: `/kaggle/input/<your-dataset>/clay_upernet_metal_roof.onnx`.
2. **Hugging Face Hub** — push the model to a private repo and pull via `hf_hub_download`.
3. **Skip inference** — set `MODEL_PATH = None` to stop after data acquisition (useful for batch data prep).

In [ ]:
import onnxruntime as ort

# --- Locate model -----------------------------------------------------------
MODEL_PATH = None
DEPLOY_CFG = None

# Search /kaggle/input/* for an .onnx + deploy_cfg.json pair
for cand in Path("/kaggle/input").glob("**/*.onnx"):
    cfg_path = cand.parent / "deploy_cfg.json"
    if cfg_path.exists():
        MODEL_PATH = cand
        DEPLOY_CFG = json.loads(cfg_path.read_text())
        break
if MODEL_PATH is None:
    # Try /kaggle/working as a fallback (if you're running both notebooks in one session)
    for cand in Path("/kaggle/working").glob("**/*.onnx"):
        cfg_path = cand.parent / "deploy_cfg.json"
        if cfg_path.exists():
            MODEL_PATH = cand
            DEPLOY_CFG = json.loads(cfg_path.read_text())
            break

if MODEL_PATH is None:
    print("ℹ No ONNX model found. Skipping inference.\n"
          "  Attach your training-notebook output as a Kaggle Dataset to enable.")
else:
    print(f"✓ Model:       {MODEL_PATH}")
    print(f"✓ deploy_cfg:  {DEPLOY_CFG}")


In [ ]:
if MODEL_PATH is not None:
    def torch_cuda_available():
        try:
            import torch; return torch.cuda.is_available()
        except Exception: return False

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch_cuda_available() else ["CPUExecutionProvider"]
    sess = ort.InferenceSession(str(MODEL_PATH), providers=providers)
    print(f"ORT providers: {sess.get_providers()}")
    print(f"  inputs:  {[(i.name, i.shape, i.type) for i in sess.get_inputs()]}")
    print(f"  outputs: {[(o.name, o.shape, o.type) for o in sess.get_outputs()]}")


In [ ]:
# --- Preprocess: data has 11 bands (10 spectral + SCL); model expects 3 (RGB) ---
# Our trained model was trained on RGB tiles. To use the multispectral signal we
# would need to retrain. For now, derive RGB from B04/B03/B02 (true colour).
# Apply the same ImageNet normalization as the training notebook.

if MODEL_PATH is not None:
    mean = np.array(DEPLOY_CFG["mean"], dtype=np.float32)
    std  = np.array(DEPLOY_CFG["std"],  dtype=np.float32)
    img_size = DEPLOY_CFG["img_size"]
    class_names = DEPLOY_CFG["class_names"]

    # Build RGB tile, scale to 0-1
    rgb_tile = np.stack([B["B04"], B["B03"], B["B02"]], axis=-1)
    rgb_tile = stretch(rgb_tile)   # 0-1
    if rgb_tile.shape[:2] != (img_size, img_size):
        import cv2
        rgb_tile = cv2.resize(rgb_tile, (img_size, img_size), interpolation=cv2.INTER_AREA)
    x = ((rgb_tile - mean) / std).astype(np.float32)
    x = np.transpose(x, (2, 0, 1))[None]   # (1, 3, H, W)

    print(f"Input shape: {x.shape}, range [{x.min():.2f}, {x.max():.2f}]")

    # Forward pass
    t0 = time.time()
    out_names = [o.name for o in sess.get_outputs()]
    outputs = sess.run(out_names, {"pixel_values": x})
    out_dict = dict(zip(out_names, outputs))
    print(f"✓ Inference: {(time.time()-t0)*1000:.0f} ms")

    mat_logits = out_dict["material_logits"]   # (1, 5, H, W)
    cor_logits = out_dict["corrosion_logits"]  # (1, 1, H, W)
    sev_ord    = out_dict["severity_ordinal"]  # (1, K-1)

    mat_pred = mat_logits[0].argmax(0).astype(np.uint8)
    cor_prob = 1 / (1 + np.exp(-cor_logits[0, 0]))
    cor_pred = (cor_prob > 0.5).astype(np.uint8)
    sev_class = int(((1 / (1 + np.exp(-sev_ord[0]))) > 0.5).sum())
    sev_label = ["none", "light", "moderate", "severe"][sev_class]

    print(f"\n=== Prediction summary ===")
    for k, n in zip(class_names, range(len(class_names))):
        share = (mat_pred == n).mean() * 100
        print(f"  {k:20}: {share:5.1f}% of tile")
    print(f"  corrosion (any) : {cor_pred.mean()*100:5.1f}% of tile")
    print(f"  severity class  : {sev_class} ({sev_label})")


## 9. Per-polygon quotation calculation

Convert pixel statistics → m² → THB.

### Cost table (2024–2025 baseline, refresh quarterly)

Sources: Bureau of the Budget standard cost catalogue (yotathai.com), Thai Valuers Association property-tax bulletin, retail prices (Bluescope, Thaiwatsadu, HomePro). Stored as YAML so non-engineers can edit.

In [ ]:
# --- Indicative Thai pricing (THB/m²) ---------------------------------------
# These are starting points — refresh from gov.th sources quarterly.
PRICING_THB_PER_M2 = {
    "corrugated_metal":     {"material": 490, "labour": 200, "total_min": 450, "total_max": 750},
    "painted_metal":        {"material": 580, "labour": 220, "total_min": 550, "total_max": 950},
    "standing_seam_metal":  {"material": 720, "labour": 250, "total_min": 700, "total_max": 1200},
    "metal_tile":           {"material": 650, "labour": 250, "total_min": 700, "total_max": 1100},
    "other":                {"material": 0,   "labour": 0,   "total_min": 0,   "total_max": 0},
}
CORROSION_REPLACEMENT_FACTOR = {0: 0.0, 1: 0.30, 2: 0.65, 3: 1.0}   # severity → replace fraction


In [ ]:
def estimate_quote(mat_pred: np.ndarray, cor_pred: np.ndarray, sev_class: int,
                   tile_extent_m: tuple, pricing: dict = PRICING_THB_PER_M2,
                   class_names: list = None):
    """Returns a quote dict suitable for PDF rendering downstream."""
    H, W = mat_pred.shape
    px_area_m2 = (tile_extent_m[0] * tile_extent_m[1]) / (W * H)

    line_items = []
    total_lo, total_hi = 0.0, 0.0
    for cls_id, name in enumerate(class_names or list(pricing.keys())):
        if name == "other": continue
        cls_pixels = (mat_pred == cls_id).sum()
        cls_area = cls_pixels * px_area_m2
        if cls_area < 1.0: continue

        cor_pixels_in_cls = ((mat_pred == cls_id) & (cor_pred == 1)).sum()
        cor_share = cor_pixels_in_cls / max(cls_pixels, 1)

        # Apply severity-aware replacement factor
        repair_fraction = max(cor_share, CORROSION_REPLACEMENT_FACTOR[sev_class] * cor_share)
        repair_area = cls_area * repair_fraction

        p = pricing[name]
        item_lo = repair_area * p["total_min"]
        item_hi = repair_area * p["total_max"]
        total_lo += item_lo
        total_hi += item_hi
        line_items.append({
            "material":             name,
            "total_area_m2":        round(cls_area, 1),
            "corroded_share":       round(cor_share, 3),
            "repair_area_m2":       round(repair_area, 1),
            "unit_price_low_THB":   p["total_min"],
            "unit_price_high_THB":  p["total_max"],
            "subtotal_low_THB":     round(item_lo, 0),
            "subtotal_high_THB":    round(item_hi, 0),
        })

    return {
        "tile_extent_m":      [round(tile_extent_m[0], 1), round(tile_extent_m[1], 1)],
        "tile_area_m2":       round(tile_extent_m[0] * tile_extent_m[1], 0),
        "severity_class":     sev_class,
        "severity_label":     ["none", "light", "moderate", "severe"][sev_class],
        "line_items":         line_items,
        "total_low_THB":      round(total_lo, 0),
        "total_high_THB":     round(total_hi, 0),
        "confidence_band":    "±15% at 95% CI (Sentinel-2 Tier-0; escalate to VHR for binding quote)",
        "model_version":      DEPLOY_CFG.get("decoder", "?") if DEPLOY_CFG else "?",
        "imagery_source":     f"Sentinel-2 L2A {acquisition_date} ({best_scene['properties']['eo:cloud_cover']:.1f}% cloud)",
        "scene_id":           best_scene["id"],
    }

if MODEL_PATH is not None:
    quote = estimate_quote(mat_pred, cor_pred, sev_class,
                            tile_extent_m=(ground_w_m, ground_h_m),
                            class_names=class_names)
    print(json.dumps(quote, indent=2, ensure_ascii=False))

    quote_path = OUT_DIR / f"{aoi_name}_{acquisition_date}_quote.json"
    quote_path.write_text(json.dumps(quote, indent=2, ensure_ascii=False))
    print(f"\n✓ Quote → {quote_path}")


In [ ]:
# --- Visualise prediction overlay ------------------------------------------
if MODEL_PATH is not None:
    PALETTE = np.array([
        [128,128,128], [40, 90,200], [140,150,180], [180, 60, 40], [80, 80, 80],
    ], dtype=np.uint8)

    def color_mat(m):
        out = np.zeros((*m.shape, 3), dtype=np.uint8)
        for k in range(len(PALETTE)): out[m == k] = PALETTE[k]
        return out

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(rgb_tile); axes[0].set_title("RGB input"); axes[0].axis("off")
    axes[1].imshow(color_mat(mat_pred)); axes[1].set_title("Material"); axes[1].axis("off")
    axes[2].imshow(cor_prob, cmap="hot", vmin=0, vmax=1); axes[2].set_title(f"Corrosion prob (sev={sev_label})"); axes[2].axis("off")
    overlay = rgb_tile.copy(); overlay[cor_pred == 1] = [1, 0.1, 0.1]
    axes[3].imshow(np.clip(0.55*rgb_tile + 0.45*overlay, 0, 1)); axes[3].set_title("Overlay"); axes[3].axis("off")

    import matplotlib.patches as mpatches
    handles = [mpatches.Patch(color=PALETTE[k]/255, label=class_names[k]) for k in range(5)]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=5)
    plt.tight_layout(); plt.show()


## 10. Batch pipeline — multiple AOIs, training data prep

The same ingestion pipeline, wrapped to iterate over a FeatureCollection of polygons. Use this to:
- **Build a Thai training set**: 100 polygons across Bangkok / Chiang Mai / Khon Kaen / Hat Yai
- **Refresh customer cohort**: re-image the same polygons monthly for change detection
- **Pre-compute Tier-0 quotes** for an estimator's day's worth of customer requests

In [ ]:
def fetch_one_aoi(feature: dict, days_back: int = 90, max_cloud: float = 20.0,
                  out_dir: Path = OUT_DIR) -> dict:
    """End-to-end for a single AOI feature. Returns metadata."""
    name = feature.get("properties", {}).get("name", f"aoi_{int(time.time())}")
    geom = shape(feature["geometry"])
    out = {"name": name, "status": "pending", "error": None}

    try:
        scenes = search_s2_scenes(geom, days_back=days_back, max_cloud=max_cloud)
        if not scenes:
            out["status"] = "no_scenes"; return out

        s = scenes[0]
        utm = utm_epsg_for_geom(geom)
        ut = Transformer.from_crs(4326, utm, always_xy=True).transform
        gu = shp_transform(ut, geom)
        bb = BBox(bbox=gu.bounds, crs=CRS(utm))
        date = s["properties"]["datetime"][:10]

        req = SentinelHubRequest(
            evalscript=EVALSCRIPT_S2_FULL,
            input_data=[SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(date, date),
                mosaicking_order=MosaickingOrder.LEAST_CC,
                maxcc=0.6,
            )],
            responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
            bbox=bb, size=(384, 384), config=config,
        )
        arr = req.get_data()[0]

        tp = out_dir / f"{name}_{date}_s2l2a.tif"
        tr = from_bounds(*gu.bounds, 384, 384)
        with rasterio.open(tp, "w", driver="COG", width=384, height=384, count=11,
                           dtype="float32", crs=f"EPSG:{utm}", transform=tr,
                           compress="zstd", blocksize=256) as dst:
            for i, n in enumerate(band_names):
                dst.write(arr[..., i].astype("float32"), i + 1)
                dst.set_band_description(i + 1, n)
            dst.update_tags(scene_id=s["id"], cloud_cover=str(s["properties"]["eo:cloud_cover"]))

        out.update(
            status="ok",
            scene_id=s["id"],
            date=date,
            cloud_cover=s["properties"]["eo:cloud_cover"],
            tile_path=str(tp),
            ground_w_m=round(gu.bounds[2]-gu.bounds[0], 1),
            ground_h_m=round(gu.bounds[3]-gu.bounds[1], 1),
        )
    except Exception as e:
        out["status"] = "error"; out["error"] = f"{type(e).__name__}: {e}"
    return out

def fetch_all(feature_collection: dict, max_cloud: float = 20.0):
    from tqdm.auto import tqdm
    feats = (feature_collection["features"]
             if feature_collection["type"] == "FeatureCollection" else [feature_collection])
    rows = []
    for ft in tqdm(feats):
        rows.append(fetch_one_aoi(ft, max_cloud=max_cloud))
    return rows

# Demo: just re-fetch the example AOI, but in batch form
batch_results = fetch_all({"type": "FeatureCollection", "features": [EXAMPLE_AOI]})
print(json.dumps(batch_results, indent=2)[:1000])


## 11. Operational notes

### CDSE rate limits (2026)
- **Free tier**: 30,000 Processing Units/month, ~500 PU/min burst.
- One 384×384 11-band L2A request ≈ 4 PU. Budget ~7,500 requests/month free.
- 429 errors → exponential backoff. `sentinelhub-py` does this automatically.
- Hard cap exceeded → upgrade to Copernicus Contributing Mission tier (paid) or move bulk pulls to direct S3 via `eodata.dataspace.copernicus.eu` (free for the actual data, you pay for compute).

### Cloud cover in Thailand
- **Dry season (Nov–Apr)**: median cloud cover ~20–35%, plenty of <20% scenes.
- **Wet season (May–Oct)**: median 70–85%, expect <5 cloud-free scenes per quarter.
  - Always set `days_back=90` minimum, fall back to 180.
  - Use Best-Available-Pixel composite (CDSE openEO recipe) for systematic coverage.
  - Trigger Sentinel-1 SAR fusion for missing tiles (separate notebook).

### Cost tracking
Add a wrapper that logs estimated PU cost per request to Postgres so you can correlate spend with revenue per quote. Sentinel Hub returns the actual PU charged in the `x-processingunits-spent` response header — capture it.

### Production hardening
1. **Token caching**: tokens valid 10 min; cache and refresh on 401 instead of fetching every request.
2. **Retry with exponential backoff** on 5xx, 429.
3. **Idempotency**: hash `(polygon, scene_id, evalscript)` → cache key in Cloudflare R2; skip re-fetch.
4. **Parallel batch**: `concurrent.futures.ThreadPoolExecutor(max_workers=4)` — `sentinelhub-py` is thread-safe and connection-pooled. Don't go above 4 workers without explicit rate-budget arithmetic.
5. **Audit trail**: persist `(request_payload, response_metadata, model_version, mask_uri, quote_json)` to Postgres + R2 for 7-year retention (insurance regulatory norm — see Troutman 2024 NAIC bulletin).

### When to escalate beyond Sentinel-2
| Trigger | Escalation |
|---|---|
| Polygon area < 800 m² (single building) | Pléiades archive (€3.80/km², 50 cm) |
| Quote value > THB 100k | Pléiades Neo tasking (€18/km², 30 cm, 12 h) |
| Customer disputes auto-quote | CAAT-licensed drone partner, 3–5 cm GSD |
| Cloud cover > 60% in last 90 d | S1 SAR triage + paid VHR archive |
| Asbestos suspected | Manual estimator inspection (S2 cannot disambiguate AC from cement) |

### Where this notebook fits in the bigger architecture

```
[Vercel Frontend] ── polygon ──▶ [RunPod FastAPI handler]
                                    │
                                    ▼
                              this notebook's logic
                              (vendored as a Python module)
                                    │
                                    ▼
                              [CDSE Sentinel Hub]
                                    │
                                    ▼
                              ONNX inference on L40S
                                    │
                                    ▼
                              Quote → Supabase + R2 + email
```

The full architecture and economics are documented in the design report from earlier in the conversation.

🛰️🏠⚡ Ship safely. Rotate credentials. Don't paste secrets.